In [7]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

filepath = "D:\BDA 20\ABD-BDA-LabWork\datasets\sf-fire-calls.csv"

In [8]:
def create_session():
    spark = SparkSession.builder.appName('Fire Example').getOrCreate()
    return spark

In [9]:
#loads dataset to dataframes
#call type = key here

#header = True    ---- it means there is header so, ignore first row and consider it as column name.
# inferSchema = True         --- we asked to pyspark, to fgured out schema by himself.


def create_df(spark, filepath):
    df = spark.read.csv(filepath, header = True, inferSchema = True)
    df1 = df.select('CallType', 'CallDate', 'City','Zipcode','Neighborhood','Delay')         #we filter column that we needed 
    return df1


In [10]:
def clean_dataSet(df):
    df1 = df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2 = df1.withColumn('Year',year(col('Date')))\
    .withColumn('Month',month(col('Date')))\
    .withColumn('Week',weekofyear(col('Date')))

    return df2

In [11]:
spark = create_session()
df = create_df(spark, filepath)
df = clean_dataSet(df)
df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [12]:
df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [13]:
#Create a user defined function

def mapSession(data):
    if 2 < data < 6:
        return 'Spring'
    elif 5 < data < 9:
        return 'Summer'
    elif 8 < data < 12:
        return 'Autumn'
    else:
        return 'Winter'

seasonUDF = udf(mapSession, StringType())
clean = df.withColumn('Season', seasonUDF(col('Month')))
clean.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102